In [1]:
!pip install ablang2
!pip install torch
!pip install AbLang
!pip install anarci
#%load_ext autoreload
#%autoreload 2

import numpy as np
import torch
import ablang2
from IPython.display import display

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 37.6 MB/s eta 0:00:00


# **0. Sequence input and its format**

AbLang2 takes as input either the individual heavy variable domain (VH), light variable domain (VL), or the full variable domain (Fv).

Each record (antibody) needs to be a list with the VH as the first element and the VL as the second. If either the VH or VL is not known, leave an empty string.

An asterisk (\*) is used for masking. It is recommended to mask residues which you are interested in mutating.

**NB:** It is important that the VH and VL sequence is ordered correctly.

In [2]:
seq1 = [
    'EVQLLESGGEVKKPGASVKVSCRASGYTFRNYGLTWVRQAPGQGLEWMGWISAYNGNTNYAQKFQGRVTLTTDTSTSTAYMELRSLRSDDTAVYFCARDVPGHGAAFMDVWGTGTTVTVSS', # VH sequence
    'DIQLTQSPLSLPVTLGQPASISCRSSQSLEASDTNIYLSWFQQRPGQSPRRLIYKISNRDSGVPDRFSGSGSGTHFTLRISRVEADDVAVYYCMQGTHWPPAFGQGTKVDIK' # VL sequence
]
seq2 = [
    'EVQLLESGGEVKKPGASVKVSCRASGYTFRNYGLTWVRQAPGQGLEWMGWISAYNGNTNYAQKFQGRVTLTTDTSTSTAYMELRSLRSDDTAVYFCARDVPGHGAAFMDVWGTGTT',
    'PVTLGQPASISCRSSQSLEASDTNIYLSWFQQRPGQSPRRLIYKISNRDSGVPDRFSGSGSGTHFTLRISRVEADDVAVYYCMQGTHWPPAFGQGTKVDIK'
]
seq3 = [
    'EVQLLESGGEVKKPGASVKVSCRASGYTFRNYGLTWVRQAPGQGLEWMGWISAYNGNTNYAQKFQGRVTLTTDTSTSTAYMELRSLRSDDTAVYFCARDVPGHGAAFMDVWGTGTTVTVSS',
    '' # The VL sequence is not known, so an empty string is left instead.
]
seq4 = [
    '',
    'DIQLTQSPLSLPVTLGQPASISCRSSQSLEASDTNIYLSWFQQRPGQSPRRLIYKISNRDSGVPDRFSGSGSGTHFTLRISRVEADDVAVYYCMQGTHWPPAFGQGTKVDIK'
]
seq5 = [
    'EVQ***SGGEVKKPGASVKVSCRASGYTFRNYGLTWVRQAPGQGLEWMGWISAYNGNTNYAQKFQGRVTLTTDTSTSTAYMELRSLRSDDTAVYFCAR**PGHGAAFMDVWGTGTTVTVSS', # (*) is used to mask certain residues
    'DIQLTQSPLSLPVTLGQPASISCRSS*SLEASDTNIYLSWFQQRPGQSPRRLIYKI*NRDSGVPDRFSGSGSGTHFTLRISRVEADDVAVYYCMQGTHWPPAFGQGTKVDIK'
]

all_seqs = [seq1, seq2, seq3, seq4, seq5]
only_both_chains_seqs = [seq1, seq2, seq5]

# **1. How to use AbLang2**

AbLang2 can be downloaded and used in its raw form as seen below. For convenience, we have also developed different "modes" which can be used for specific use cases (see Section 2)

In [3]:
# Download and initialise the model
ablang = ablang2.pretrained(model_to_use='ablang2-paired', random_init=False, ncpu=1, device='cpu')

# Tokenize input sequences
seq = f"{seq1[0]}|{seq1[1]}" # VH first, VL second, with | used to separated the two sequences
tokenized_seq = ablang.tokenizer([seq], pad=True, w_extra_tkns=False, device="cpu")

# Generate rescodings
with torch.no_grad():
    rescoding = ablang.AbRep(tokenized_seq).last_hidden_states

# Generate logits/likelihoods
with torch.no_grad():
    likelihoods = ablang.AbLang(tokenized_seq)

# **2. Different modes for specific usecases**

AbLang2 has already been implemented for a variety of different usecases. The benefit of these modes is that they handle extra tokens such as start, stop and separation tokens.

1. seqcoding: Generates sequence representations for each sequence
2. rescoding: Generates residue representations for each residue in each sequence
3. likelihood: Generates likelihoods for each amino acid at each position in each sequence
4. probability: Generates probabilities for each amino acid at each position in each sequence
5. pseudo_log_likelihood: Returns the pseudo log likelihood for a sequence (based on masking each residue one at a time)
6. confidence: Returns a fast calculation of the log likelihood for a sequence (based on a single pass with no masking)
7. restore: Restores masked residues

### **AbLang2 can also align the resulting representations using ANARCI**

This can be done for 'rescoding', 'likelihood', and 'probability'. This is done by setting the argument "align=True".

**NB**: Align can only be used on input with the same format, i.e. either all heavy, all light, or all both heavy and light.

### **The align argument can also be used to restore variable missing lengths**

For this, use "align=True" with the 'restore' mode.

In [4]:
ablang = ablang2.pretrained()

valid_modes = [
    'seqcoding', 'rescoding', 'likelihood', 'probability',
    'pseudo_log_likelihood', 'confidence', 'restore'
]

## **seqcoding**

The seqcodings represents each sequence as a 480 sized embedding. It is derived from averaging across each rescoding embedding for a given sequence, including extra tokens.

**NB:** Seqcodings can also be derived in other ways like using the sum or averaging across only parts of the input such as the CDRs. For such cases please use and adapt the below rescoding.

In [5]:
ablang(all_seqs, mode='probability')

[array([[9.9955505e-01, 2.9358694e-06, 2.4716180e-06, ..., 3.5776267e-11,
         9.1196654e-08, 7.0967457e-05],
        [4.1573694e-06, 4.1619370e-07, 2.5800991e-06, ..., 3.4650885e-10,
         1.0159109e-08, 1.6279498e-06],
        [7.8059443e-08, 1.2794004e-03, 4.0645340e-05, ..., 3.0375715e-09,
         1.8879488e-10, 4.2010754e-08],
        ...,
        [3.4210811e-07, 1.7195308e-03, 1.8477205e-05, ..., 1.5137445e-10,
         6.4255751e-10, 8.2064140e-08],
        [9.1038261e-09, 4.5161933e-06, 5.4411950e-05, ..., 9.1139804e-11,
         6.0861971e-08, 8.1761815e-09],
        [8.5759582e-04, 2.0104915e-05, 3.9023766e-06, ..., 4.5562460e-11,
         8.1156777e-08, 5.4990756e-05]], dtype=float32),
 array([[9.9939799e-01, 4.1499175e-06, 1.8611148e-05, ..., 1.8139243e-10,
         5.5649405e-08, 1.2583827e-04],
        [1.6735529e-06, 5.4332202e-07, 3.4143536e-06, ..., 3.1693398e-10,
         7.1501400e-09, 9.6832684e-07],
        [3.7784993e-08, 1.2377622e-04, 1.7658769e-05, ...,

## **rescoding / likelihood / probability**

The rescodings represents each residue as a 480 sized embedding. The likelihoods represents each residue as the predicted logits for each character in the vocabulary. The probabilities represents the normalised likelihoods.

**NB:** The output includes extra tokens (start, stop and separation tokens) in the format "<VH_seq>|<VL_seq>". The length of the output is therefore 5 longer than the VH and VL.

**NB:** By default the representations are derived using a single forward pass. To prevent the predicted likelihood and probability to be affected by the input residue at each position, setting the "stepwise_masking" argument to True can be used. This will run a forward pass for each position with the residue at that position masked. This is much slower than running a single forward pass.

In [6]:
ablang(all_seqs, mode='rescoding', stepwise_masking = False)

[array([[-0.40741193, -0.5118987 ,  0.06096716, ...,  0.32681432,
          0.03920236, -0.36715853],
        [-0.57688814,  0.38245395, -0.21791996, ...,  0.01250292,
         -0.08844497, -0.32367525],
        [-0.14759341,  0.3963903 , -0.3822692 , ..., -0.10119932,
         -0.41469547, -0.00319354],
        ...,
        [-0.14358398,  0.3124389 , -0.30157974, ..., -0.13289282,
         -0.45353428, -0.07878865],
        [ 0.17538951,  0.24394295,  0.20141171, ...,  0.14587368,
         -0.38479006,  0.07409185],
        [-0.2303168 , -0.3548732 ,  0.19606812, ..., -0.1283364 ,
          0.31107324, -0.32651082]], dtype=float32),
 array([[-0.4198181 , -0.36663747,  0.10595222, ...,  0.39035732,
          0.03823784, -0.36337987],
        [-0.50541365,  0.38347092, -0.10992126, ..., -0.05231531,
         -0.13636588, -0.3483009 ],
        [-0.06784617,  0.69349897, -0.4212398 , ..., -0.24805355,
         -0.3958379 , -0.10972734],
        ...,
        [-0.20900992,  0.29489496, -0.1

## **Align rescoding/likelihood/probability output**

For the 'rescoding', 'likelihood', and 'probability' modes, the output can also be aligned using the argument "align=True".

This is done using the antibody numbering tool ANARCI, and requires manually installing **Pandas** and **[ANARCI](https://github.com/oxpig/ANARCI)**.

**NB**: Align can only be used on input with the same format, i.e. either all heavy, all light, or all both heavy and light.

In [7]:
results = ablang(only_both_chains_seqs, mode='likelihood', align=True)

print(results.number_alignment)
print(results.aligned_seqs)
print(results.aligned_embeds)

FileNotFoundError: [Errno 2] No such file or directory: 'hmmscan'

## **Pseudo log likelihood and Confidence scores**

The pseudo log likelihood and confidence represents two methods for calculating the uncertainty for the input sequence.

- pseudo_log_likelihood: For each position, the pseudo log likelihood is calculated when predicting the masked residue. The final score is an average across the whole input. This is similar to the approach taken in the ESM-2 paper for calculating pseudo perplexity [(Lin et al., 2023)](https://doi.org/10.1126/science.ade2574).

- confidence: For each position, the log likelihood is calculated without masking the residue. The final score is an average across the whole input.

**NB:** The **confidence is fast** to compute, requiring only a single forward pass per input. **Pseudo log likelihood is slow** to calculate, requiring L forward passes per input, where L is the length of the input.

**NB:** It is recommended to use **pseudo log likelihood for final results** and **confidence for exploratory work**.

In [ ]:
results = ablang(all_seqs, mode='pseudo_log_likelihood')
np.exp(-results) # convert to pseudo perplexity

In [ ]:
results = ablang(all_seqs, mode='confidence')
np.exp(-results)

## **restore**

This mode can be used to restore masked residues, and fragmented regions with "align=True".

In [ ]:
restored = ablang(only_both_chains_seqs, mode='restore')
restored

In [ ]:
restored = ablang(only_both_chains_seqs, mode='restore', align = True)
restored

## Unzip and Process Masked Sequences

First, I will unzip the `masked_ablang_2.zip` file to access the CSVs containing the masked sequences and their corresponding true amino acids.

In [8]:
import zipfile
import os
import pandas as pd
import torch
import numpy as np

# Define the path to the zip file and the extraction directory
zip_file_path = '/content/masked_ablang_2.zip'
extract_dir_base = '/content/masked_ablang_data'

# Create the extraction directory if it doesn't exist
os.makedirs(extract_dir_base, exist_ok=True)

# Check if the zip file exists
if not os.path.exists(zip_file_path):
    print(f"Error: Zip file not found at '{zip_file_path}'")
    csv_files = [] # Ensure csv_files is empty to prevent further errors
else:
    # Unzip the file
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir_base)

    print(f"'{zip_file_path}' unzipped to '{extract_dir_base}' successfully.")

    # Check for a nested directory with the same name as the zip file (without extension)
    nested_dir_name = os.path.splitext(os.path.basename(zip_file_path))[0]
    potential_nested_dir_path = os.path.join(extract_dir_base, nested_dir_name)

    if os.path.isdir(potential_nested_dir_path):
        # If the nested directory exists, use it as the actual directory for CSV searching
        actual_csv_search_dir = potential_nested_dir_path
        print(f"Found nested directory '{nested_dir_name}'. Searching for CSVs within it.")
    else:
        # Otherwise, search directly in the base extraction directory
        actual_csv_search_dir = extract_dir_base
        print(f"No nested directory '{nested_dir_name}' found. Searching for CSVs in '{extract_dir_base}'.")

    # List CSV files in the determined directory. Search recursively.
    csv_files = []
    for root, _, files in os.walk(actual_csv_search_dir):
        for file in files:
            if file.endswith('.csv'):
                csv_files.append(os.path.join(root, file))

    if not csv_files:
        print(f"No CSV files found in '{actual_csv_search_dir}' or its subdirectories.")
    else:
        print(f"Found {len(csv_files)} CSV files:")
        for f in csv_files:
            print(f"  - {f}")

'/content/masked_ablang_2.zip' unzipped to '/content/masked_ablang_data' successfully.
Found nested directory 'masked_ablang_2'. Searching for CSVs within it.
Found 2236 CSV files:
  - /content/masked_ablang_data/masked_ablang_2/Bapineuzumab_K_K.csv
  - /content/masked_ablang_data/masked_ablang_2/Brontictuzumab_L_L.csv
  - /content/masked_ablang_data/masked_ablang_2/Firsekibart_H_H.csv
  - /content/masked_ablang_data/masked_ablang_2/Gatipotuzumab_H_H.csv
  - /content/masked_ablang_data/masked_ablang_2/Danvilostomig_K_K.csv
  - /content/masked_ablang_data/masked_ablang_2/Orticumab_L_L.csv
  - /content/masked_ablang_data/masked_ablang_2/Lifastuzumab_H_H.csv
  - /content/masked_ablang_data/masked_ablang_2/Felmetatug_H_H.csv
  - /content/masked_ablang_data/masked_ablang_2/Etigilimab_K_K.csv
  - /content/masked_ablang_data/masked_ablang_2/Landogrozumab_K_K.csv
  - /content/masked_ablang_data/masked_ablang_2/Opicinumab_K_K.csv
  - /content/masked_ablang_data/masked_ablang_2/Azerutamig_K_K.cs

### Assumed CSV Structure

For each CSV file, I will assume the following structure:

*   **Row 1**: Contains the `chain_type` ('H' for Heavy, 'K' for Light) in the first column and the `full_masked_sequence_string` in the second column.
    *   Example: `H,EVQLLESGGEVKKPGASVKVSCRASGYTFRNYGLTWVRQAPGQGLEWMGWISAYNGNTNYAQKFQGRVTLTTDTSTSTAYMELRSLRSDDTAVYFCAR**PGHGAAFMDVWGTGTTVTVSS`

*   **Subsequent Rows**: Contain `masked_token_position` (0-indexed position in the sequence string) and `actual_amino_acid` for each masked token in the sequence.
    *   Example:
        `position,aa`
        `84,D`
        `85,D`

This structure allows me to extract the full masked sequence and then iterate through the specific masked positions and their true amino acid values for comparison with `ablang2` predictions.

In [10]:
from ablang2 import pretrained # Ensure ablang2 is loaded as in previous cells

# Initialize AbLang2 model if not already done
ablang = ablang2.pretrained(model_to_use='ablang2-paired', random_init=False, ncpu=1, device='cpu')

results_data = []

if not csv_files:
    print("No CSV files to process. Please ensure the zip file contains CSVs and try again.")
else:
    # Determine the token-to-index mapping strategy once
    token_id_to_aa_fn = None

    # Debugging: Print available attributes of the tokenizer
    print(f"ablang.tokenizer type: {type(ablang.tokenizer)}")
    print(f"ablang.tokenizer dir: {dir(ablang.tokenizer)}")

    # Prioritize 'token_to_aa' as it was identified in dir()
    if hasattr(ablang.tokenizer, 'token_to_aa'): # Using the identified attribute
        token_id_to_aa_fn = lambda idx: ablang.tokenizer.token_to_aa[idx] # Assuming token_to_aa is a list-like mapping
    # Then try other common methods
    elif hasattr(ablang.tokenizer, 'convert_ids_to_tokens') and callable(ablang.tokenizer.convert_ids_to_tokens):
        token_id_to_aa_fn = lambda idx: ablang.tokenizer.convert_ids_to_tokens([idx])[0]
    elif hasattr(ablang.tokenizer, 'vocab'):
        # If .vocab is a dict (token: id), invert it
        if isinstance(ablang.tokenizer.vocab, dict):
            idx_to_token_map = {v: k for k, v in ablang.tokenizer.vocab.items()}
            token_id_to_aa_fn = lambda idx: idx_to_token_map[idx]
        else:
            # If .vocab is a list/tuple of tokens (idx: token)
            idx_to_token_map = ablang.tokenizer.vocab
            token_id_to_aa_fn = lambda idx: idx_to_token_map[idx]
    elif hasattr(ablang.tokenizer, 'get_vocab') and callable(ablang.tokenizer.get_vocab):
        vocab = ablang.tokenizer.get_vocab()
        if isinstance(vocab, dict):
            idx_to_token_map = {v: k for k, v in vocab.items()}
            token_id_to_aa_fn = lambda idx: idx_to_token_map[idx]
        else:
            idx_to_token_map = vocab
            token_id_to_aa_fn = lambda idx: idx_to_token_map[idx]
    elif hasattr(ablang.tokenizer, 'idx_to_token'):
        token_id_to_aa_fn = lambda idx: ablang.tokenizer.idx_to_token[idx]
    elif hasattr(ablang.tokenizer, 'ids_to_tokens'):
        token_id_to_aa_fn = lambda idx: ablang.tokenizer.ids_to_tokens[idx]
    elif hasattr(ablang.tokenizer, 'token_to_idx'):
        inverted_token_map = {v: k for k, v in ablang.tokenizer.token_to_idx.items()}
        token_id_to_aa_fn = lambda idx: inverted_token_map[idx]
    elif hasattr(ablang.tokenizer, 'all_tokens'): # Fallback for tokenizers that expose tokens as a list by ID
        token_id_to_aa_fn = lambda idx: ablang.tokenizer.all_tokens[idx]
    elif hasattr(ablang.tokenizer, 'alphabet'): # Adding 'alphabet' as a new attempt
        token_id_to_aa_fn = lambda idx: ablang.tokenizer.alphabet[idx] # Assuming alphabet is list-like indexed by token_id
    else:
        raise AttributeError("ablang.tokenizer object has no recognized vocabulary mapping attribute or method (tried 'token_to_aa', 'convert_ids_to_tokens', 'vocab', 'get_vocab', 'idx_to_token', 'ids_to_tokens', 'token_to_idx', 'all_tokens', 'alphabet'). Cannot determine vocabulary.")

    for csv_file_path in csv_files: # Use csv_file_path as it now contains the full path
        try:
            # Read the CSV, assuming the first row is the header
            df = pd.read_csv(csv_file_path)

            # Extract chain type from the 'chain' column (first non-null data row)
            chain_type = df.iloc[:, 0].dropna().iloc[0]

            # Construct the full masked sequence by joining the 'masked' column values
            full_masked_sequence = "".join(df['masked'].astype(str).tolist())

            # Find all 0-indexed positions that are masked ('*') in the full_masked_sequence
            masked_indices_in_sequence = [i for i, char in enumerate(full_masked_sequence) if char == '*']

            num_tokens_predicted = 0
            num_germline_predictions = 0

            # Prepare sequence for AbLang2 input
            if chain_type == 'H':
                ablang_input = [full_masked_sequence, '']
            elif chain_type == 'K' or chain_type == 'L': # Assuming 'L' for Light chain is also possible
                ablang_input = ['', full_masked_sequence]
            else:
                print(f"Warning: Unknown chain type '{chain_type}' in {csv_file_path}. Skipping.")
                continue

            # Run AbLang2 in 'probability' mode
            # The input needs to be a list of sequences, even if there's only one
            with torch.no_grad():
                # ablang returns a list of arrays for probability mode, one for each input sequence
                probabilities_output = ablang([ablang_input], mode='probability')

            # Since we passed a single sequence, get the first (and only) item
            sequence_probabilities = probabilities_output[0]

            for masked_seq_idx in masked_indices_in_sequence:
                # AbLang2's probability array has extra tokens (e.g., start token at index 0)
                # So, a 0-indexed position in the sequence corresponds to prob_index + 1
                prob_index = masked_seq_idx + 1

                # Ensure prob_index is within bounds
                if prob_index < sequence_probabilities.shape[0]:
                    # Get the probability distribution for the masked position
                    position_probs = sequence_probabilities[prob_index]

                    # Find the amino acid with the highest probability
                    predicted_aa_idx = np.argmax(position_probs).item() # .item() to get scalar from tensor
                    predicted_aa = token_id_to_aa_fn(predicted_aa_idx)

                    # Get the germline amino acid for this masked position from the 'aa' column
                    # This assumes a direct 0-indexed correspondence between the full_masked_sequence and df['germline']
                    true_aa = df['germline_aa'].iloc[masked_seq_idx]

                    num_tokens_predicted += 1
                    if predicted_aa == true_aa:
                        num_germline_predictions += 1
                else:
                    print(f"Warning: Masked sequence index {masked_seq_idx} (prob_index {prob_index}) for {os.path.basename(csv_file_path)} is out of bounds for probability array. Sequence length: {len(full_masked_sequence)}, Probabilities shape: {sequence_probabilities.shape[0]}")

            prediction_accuracy_ratio = (num_germline_predictions / num_tokens_predicted) if num_tokens_predicted > 0 else 0.0

            results_data.append({
                'Name': os.path.basename(csv_file_path),
                'Sequence Type': chain_type,
                'Number of Tokens Predicted': num_tokens_predicted,
                'Number of Germline Predictions': num_germline_predictions,
                'Prediction Accuracy Ratio': round(prediction_accuracy_ratio, 4)
            })
        except Exception as e:
            print(f"Error processing file {csv_file_path}: {e}")

# Create a DataFrame from the results
results_df = pd.DataFrame(results_data)

# Only calculate totals and averages if there are results
if not results_df.empty:
    # Calculate totals and averages for the final row
    total_tokens = results_df['Number of Tokens Predicted'].sum()
    total_germline_predictions = results_df['Number of Germline Predictions'].sum()
    average_accuracy_ratio = results_df['Prediction Accuracy Ratio'].mean()

    # Add the totals and averages row
    totals_row = {
        'Name': 'Total/Average',
        'Sequence Type': 'All',
        'Number of Tokens Predicted': total_tokens,
        'Number of Germline Predictions': total_germline_predictions,
        'Prediction Accuracy Ratio': round(average_accuracy_ratio, 4)
    }
    results_df = pd.concat([results_df, pd.DataFrame([totals_row])], ignore_index=True)

    # Output to CSV
    output_csv_path = '/content/ablang2_prediction_summary.csv'
    results_df.to_csv(output_csv_path, index=False)

    print(f"Prediction summary saved to '{output_csv_path}'.")
    display(results_df)
else:
    print("No results to display or save after processing.")

ablang.tokenizer type: <class 'ablang2.models.ablang2.tokenizers.ABtokenizer'>
ablang.tokenizer dir: ['__call__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'aa_to_token', 'all_special_tokens', 'decode', 'encode', 'end_token', 'mask_token', 'pad_token', 'sep_token', 'set_vocab', 'start_token', 'token_to_aa', 'unknown_token']
Prediction summary saved to '/content/ablang2_prediction_summary.csv'.


,Name,Sequence Type,Number of Tokens Predicted,Number of Germline Predictions,Prediction Accuracy Ratio
0,Bapineuzumab_K_K.csv,K,73,9,0.1233
1,Brontictuzumab_L_L.csv,L,16,0,0.0000
2,Firsekibart_H_H.csv,H,9,8,0.8889
3,Gatipotuzumab_H_H.csv,H,15,15,1.0000
4,Danvilostomig_K_K.csv,K,65,3,0.0462
...,...,...,...,...,...
2232,Omburtamab_H_H.csv,H,32,31,0.9688
2233,Telisotuzumab_K_K.csv,K,13,2,0.1538
2234,Sirexatamab_K_K.csv,K,16,1,0.0625
2235,Mecbotamab_H_H.csv,H,28,26,0.9286
